# Capítulo 13 · Laboratório 13: sentimento, o termômetro do medo e da ganância
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance, da CFTC e da Cboe na sua máquina e guarda uma cópia local (`dados_cap13.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório mede a volatilidade histórica do Ibovespa na alta e na queda, testa o VIX e o put/call de ações da Cboe como indicadores contrários, baixa o relatório COT do real direto da API pública da CFTC, conta o TD Setup e a contagem de 13 de Tom DeMark (versão didática, com as regras públicas) e monta um índice de medo e ganância didático para o Ibovespa. Atenção: o arquivo gratuito de put/call da Cboe vai só até 2019.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import requests
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap13.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def baixa_cot():
    """Relatório COT legado (só futuros) do real brasileiro na CME, código CFTC 102741, na API pública da CFTC."""
    url = ("https://publicreporting.cftc.gov/resource/6dca-aqww.json?$limit=5000&$order=report_date_as_yyyy_mm_dd"
           "&$where=cftc_contract_market_code='102741'")
    c = pd.DataFrame(requests.get(url, timeout=120).json())
    c["data"] = pd.to_datetime(c["report_date_as_yyyy_mm_dd"])
    c = c.set_index("data").sort_index()
    cols = ["noncomm_positions_long_all", "noncomm_positions_short_all", "comm_positions_long_all",
            "comm_positions_short_all", "open_interest_all"]
    return c[cols].astype(float).loc[:FIM]


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for nome, (t, ini) in {"Ibovespa": ("^BVSP", "1995-01-01"), "S&P 500": ("^GSPC", "1990-01-01"),
                       "VIX": ("^VIX", "1990-01-01"), "Dólar": ("BRL=X", "2003-01-01")}.items():
    if nome not in D:
        D[nome] = baixa(t, ini)
confere({k: D[k] for k in ("Ibovespa", "S&P 500", "VIX", "Dólar")})
if "COT" not in D:
    D["COT"] = baixa_cot()
if "PC" not in D:                       # a Cboe parou de atualizar esse arquivo gratuito em outubro de 2019
    url = "https://cdn.cboe.com/resources/options/volume_and_call_put_ratios/equitypc.csv"
    pcr = pd.read_csv(url, skiprows=2, skipinitialspace=True, storage_options={"User-Agent": "Mozilla/5.0"})
    pcr.columns = [c.strip() for c in pcr.columns]
    D["PC"] = pcr.assign(DATE=pd.to_datetime(pcr["DATE"])).set_index("DATE")["P/C Ratio"].astype(float)
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d):
    return d[(d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])]


IBD = limpa(D["Ibovespa"]).loc["2000":]
IB = IBD["Close"]
SPX, VIX = D["S&P 500"]["Close"], D["VIX"]["Close"]
USD = D["Dólar"]["Close"]
COT = D["COT"]
print(f"Parte 1. Ibovespa desde {IB.index[0]:%Y}, S&P 500 e VIX desde {VIX.index[0]:%Y}, dólar desde {USD.index[0]:%Y};"
      f" COT do real: {len(COT)} relatórios de {COT.index[0]:%m/%Y} a {COT.index[-1]:%d/%m/%Y}.")


def frente(c, h):
    return c.shift(-h) / c - 1


def resumo(f, cond):
    x = f[cond.reindex(f.index, fill_value=False)].dropna()
    return len(x), (x.mean() if len(x) else np.nan), ((x > 0).mean() if len(x) else np.nan)

## Parte 2: volatilidade histórica

In [ ]:
RET = np.log(IB).diff()
VOL21 = RET.rolling(21).std() * np.sqrt(252) * 100           # desvio-padrão anualizado (raiz de 252)
MM200 = IB.rolling(200).mean()
ALTA = (IB > MM200) & (MM200 > MM200.shift(20))
BAIXA = (IB < MM200) & (MM200 < MM200.shift(20))
CORR = IB.pct_change(21).corr(VOL21.diff(21))
print("\nParte 2. Volatilidade de 21 pregões do Ibovespa (anualizada, desde 2000)")
print(f"   mediana em tendência de alta {nr(VOL21[ALTA].median(), 0)}% | em tendência de baixa {nr(VOL21[BAIXA].median(), 0)}% |"
      f" todos os dias {nr(VOL21.median(), 0)}%")
print(f"   correlação entre o retorno de 21 pregões e a variação da volatilidade: {nr(CORR)}")

## Parte 3: VIX contrário

In [ ]:
DF = pd.DataFrame({"spx": SPX, "vix": VIX}).dropna()
CORTE = DF["vix"].quantile(0.90)
VIXT = {}
print(f"\nParte 3. VIX desde 1990: 10% dos dias acima de {nr(CORTE, 1)}")
for h in (20, 60):
    f = frente(DF["spx"], h)
    VIXT[h] = (resumo(f, DF["vix"] > CORTE), resumo(f, DF["vix"].notna()))
    (n1, m1, p1), (n2, m2, p2) = VIXT[h]
    print(f"   S&P 500 em {h} pregões: VIX alto {ps(m1)} ({pc(p1, 0)} positivos) | qualquer dia {ps(m2)} ({pc(p2, 0)} positivos)")
real = np.log(DF["spx"]).diff().rolling(21).std().shift(-21) * np.sqrt(252) * 100
VIX_ACIMA = (DF["vix"] > real).loc[real.dropna().index].mean()
print(f"   o VIX ficou acima da volatilidade realizada nos 21 pregões seguintes em {pc(VIX_ACIMA, 0)} dos dias")

## Parte 4: COT do real

In [ ]:
LIQ = COT["noncomm_positions_long_all"] - COT["noncomm_positions_short_all"]   # especuladores, líquido comprado em real
ICOT = (LIQ - LIQ.rolling(156).min()) / (LIQ.rolling(156).max() - LIQ.rolling(156).min()) * 100   # 3 anos
SEM = USD.resample("W-TUE").last().reindex(ICOT.index, method="nearest")
F12 = SEM.shift(-12) / SEM - 1                                # dólar em 12 semanas (sobe = real fraco)
COTT = {}
print(f"\nParte 4. COT do real: último relatório ({COT.index[-1]:%d/%m/%Y}): especuladores com {nr(LIQ.iloc[-1], 0)}"
      f" contratos líquidos comprados em real; índice COT de 3 anos {nr(ICOT.iloc[-1], 0)}")
for nome, cond in (("muito comprados em real (índice acima de 90)", ICOT > 90),
                   ("muito vendidos em real (índice abaixo de 10)", ICOT < 10), ("qualquer semana", ICOT.notna())):
    COTT[nome] = resumo(F12, cond)
    n, m, p = COTT[nome]
    print(f"   {nome:<46} dólar em 12 semanas: {ps(m)} ({pc(p, 0)} das semanas com alta do dólar; {n} semanas)")

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].plot(SEM.loc["2015":], color="black")
axs[0].set_title("Dólar (semanal)")
axs[1].plot(ICOT.loc["2015":], color="gray")
for y in (10, 90):
    axs[1].axhline(y, ls="--", color="black")
axs[1].set_title("Índice COT de 3 anos dos especuladores no real")
plt.show()

## Parte 5: TD Setup

In [ ]:
def contagem(cond):
    """Barras seguidas em que a condição vale (1, 2, 3...); zera quando falha."""
    return cond.astype(int).groupby((~cond).cumsum()).cumsum()


C, H, L = IBD["Close"], IBD["High"], IBD["Low"]
CC = contagem(C < C.shift(4))                                  # compra: fechamento abaixo do de 4 barras antes
CV = contagem(C > C.shift(4))
NOVE_C, NOVE_V = CC == 9, CV == 9                              # a contagem começa depois da virada (price flip)


def aperfeicoa(nove, oposto, x):
    """Marca a barra em que o 9 fica aperfeiçoado. Compra (x = mínimas): a própria barra 9, se a mínima da barra 8 ou
    da 9 for <= as mínimas das barras 6 e 7; senão, a primeira barra seguinte com mínima <= as das barras 6 e 7.
    Venda: o mesmo com x = -máximas. Versão didática: a espera acaba se um 9 no sentido oposto se completar antes."""
    marca, alvo = np.zeros(len(x), dtype=bool), None
    for i in range(len(x)):
        if nove.iloc[i]:
            alvo = min(x[i - 3], x[i - 2])                     # barras 6 e 7
            if min(x[i - 1], x[i]) <= alvo:                    # barra 8 ou barra 9
                marca[i], alvo = True, None
        elif oposto.iloc[i]:
            alvo = None                                        # 9 no sentido oposto: desiste de esperar
        elif alvo is not None and x[i] <= alvo:                # uma barra seguinte
            marca[i], alvo = True, None
    return pd.Series(marca, nove.index)


PERF_C = aperfeicoa(NOVE_C, NOVE_V, L.to_numpy())
PERF_V = aperfeicoa(NOVE_V, NOVE_C, -H.to_numpy())
TDS = {}
print("\nParte 5. TD Setup no Ibovespa desde 2000: retorno médio depois do 9 (entre parênteses, os positivos)")
for nome, cond in (("9 de compra", NOVE_C), ("9 de compra aperfeiçoado", PERF_C), ("9 de venda", NOVE_V),
                   ("9 de venda aperfeiçoado", PERF_V), ("qualquer dia", C.notna())):
    TDS[nome] = {h: resumo(frente(C, h), cond) for h in (5, 10, 20)}
    print(f"   {nome:<26} {TDS[nome][5][0]:>5} sinais | " + " | ".join(
        f"{h} pregões {ps(TDS[nome][h][1], 2)} ({pc(TDS[nome][h][2], 0)})" for h in (5, 10, 20)))

## Parte 6: TD Countdown (13)

In [ ]:
def countdown(lado):
    """Versão didática: começa na barra 9 de um setup; conta (sem precisar ser seguido) fechamentos <= mínima de 2 barras
    antes (compra) ou >= máxima de 2 barras antes (venda); a barra 13 só vale se a mínima (máxima) dela for <= (>=) o
    fechamento da barra 8 da contagem; um setup no sentido oposto cancela a contagem."""
    nove, oposto = (NOVE_C, NOVE_V) if lado == "compra" else (NOVE_V, NOVE_C)
    sinais, ativo, n, f8, inicio = [], False, 0, None, None
    c, h, l = C.to_numpy(), H.to_numpy(), L.to_numpy()
    for i in range(2, len(C)):
        if nove.iloc[i] and not ativo:
            ativo, n, f8, inicio = True, 0, None, i - 8
        elif oposto.iloc[i] and ativo:
            ativo = False
            continue
        if not ativo:
            continue
        ok = c[i] <= l[i - 2] if lado == "compra" else c[i] >= h[i - 2]
        if not ok:
            continue
        if n < 12:
            n += 1
            if n == 8:
                f8 = c[i]
        elif (l[i] <= f8) if lado == "compra" else (h[i] >= f8):
            sinais.append((C.index[i], i - inicio + 1))
            ativo = False
    return sinais


TDC = {}
print("\nParte 6. TD Countdown de 13 no Ibovespa desde 2000 (versão didática)")
for lado in ("compra", "venda"):
    s = countdown(lado)
    datas = pd.DatetimeIndex([x for x, _ in s])
    dur = np.array([b for _, b in s])
    TDC[lado] = {"datas": datas, "dur": dur, **{h: resumo(frente(C, h), pd.Series(True, datas)) for h in (10, 20, 60)}}
    print(f"   13 de {lado}: {len(s)} sinais; do 1 do setup ao 13: mediana de {np.median(dur):.0f} pregões"
          f" (de {dur.min()} a {dur.max()})")
    print("      " + " | ".join(f"{h} pregões {ps(TDC[lado][h][1], 2)} ({pc(TDC[lado][h][2], 0)} positivos)" for h in (10, 20, 60)))
print("   qualquer dia: " + " | ".join(f"{h} pregões {ps(resumo(frente(C, h), C.notna())[1], 2)}" for h in (10, 20, 60)))

## Parte 7: medo e ganância didático

In [ ]:
def percentil(x, janela=756):
    return x.rolling(janela).rank(pct=True) * 100


MOMENTO = percentil(IB / IB.rolling(125).mean() - 1)           # Ibovespa contra a média de 125 dias (alto = ganância)
CALMA = 100 - percentil(VOL21)                                  # volatilidade baixa = ganância
REAL = 100 - percentil(USD.reindex(IB.index, method="ffill").pct_change(20))   # dólar caindo = ganância
FG = pd.concat([MOMENTO, CALMA, REAL], axis=1).mean(axis=1).dropna()
F60 = frente(IB, 60)
FGT = {}
print(f"\nParte 7. Medo e ganância didático em {FG.index[-1]:%d/%m/%Y}: {nr(FG.iloc[-1], 0)} (0 = medo extremo, 100 ="
      " ganância extrema)")
for nome, cond in (("medo extremo (abaixo de 20)", FG < 20), ("neutro (40 a 60)", FG.between(40, 60)),
                   ("ganância extrema (acima de 80)", FG > 80), ("qualquer dia", FG.notna())):
    FGT[nome] = resumo(F60, cond)
    n, m, p = FGT[nome]
    print(f"   {nome:<32} Ibovespa em 60 pregões: {ps(m)} ({pc(p, 0)} positivos; {n} dias)")

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].plot(IB.loc["2019":], color="black")
axs[1].plot(FG.loc["2019":], color="gray")
for y in (20, 80):
    axs[1].axhline(y, ls="--", color="black")
axs[1].set_title("Medo e ganância didático")
plt.show()

## Parte 8: put/call ratio

In [ ]:
PCR = D["PC"]
PC10 = PCR.rolling(10).mean()
PCT = {}
q90, q10 = PC10.quantile(0.90), PC10.quantile(0.10)
print(f"\nParte 8. Put/call de ações da Cboe ({PCR.index[0]:%m/%Y} a {PCR.index[-1]:%m/%Y}), média de 10 dias: mediana"
      f" {nr(PC10.median())}; 10% mais altos acima de {nr(q90)}; 10% mais baixos abaixo de {nr(q10)}")
for nome, cond in (("put/call alto (medo)", PC10 > q90), ("put/call baixo (otimismo)", PC10 < q10),
                   ("qualquer dia", PC10.notna())):
    PCT[nome] = {h: resumo(frente(SPX, h), cond) for h in (20, 60)}
    print(f"   {nome:<26} S&P 500 em 20 pregões {ps(PCT[nome][20][1])} ({pc(PCT[nome][20][2], 0)}) | em 60 pregões"
          f" {ps(PCT[nome][60][1])} ({pc(PCT[nome][60][2], 0)})")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `contagem` e `aperfeicoa` e as linhas da Parte 5 que criam `CC`, `NOVE_C` e `PERF_C`. Mostre onde está o `shift(4)`, quais barras `x[i - 3]` e `x[i - 2]` representam no aperfeiçoamento e onde o código trata o caso de uma barra seguinte à 9." Confira com a Parte 5, que imprime 78 sinais de 9 de compra e 72 de 9 de compra aperfeiçoado no Ibovespa desde 2000.
2. **Adaptar:** "Acrescente [ATIVO] ao dicionário da célula de abertura, com o ticker do Yahoo Finance e a data inicial. Numa célula nova, depois da Parte 5, repita as contas da Parte 5 para [ATIVO] com a função `contagem`, usando nomes novos no lugar de `C`, `H` e `L`, que a Parte 6 usa. Não mude a comparação com 4 barras antes nem a contagem de 9." Antes de olhar o ativo novo, confira se a Parte 5 continua com 78 sinais de 9 de compra no Ibovespa, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 3, troque o corte de 10% dos dias de VIX mais alto por 5%. O resultado fica melhor ou só mais raro?
2. Na Parte 4, use os comerciais em vez dos especuladores. Eles acertam mais o dólar?
3. Na Parte 5, rode o TD Setup no dólar (BRL=X) ou no bitcoin (BTC-USD). O 9 funciona melhor fora do Ibovespa?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.